In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import sys

sys.path.append('/home/wolfgang/repos/AirGo_Apnea_Detection')
from apnea_detection_functions import apply_apnea_prediction_models

sys.path.append('/home/wolfgang/repos/ICU-Sleep/code1')
from airgo_features import compute_airgo_features
from sleep_staging_functions import apply_airgo_sleep_staging_models, apply_airgo_sleep_staging_models_fold

sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file

%matplotlib widget
import matplotlib.pyplot as plt

In [2]:
def filepaths_routine(data_dir, output_dir, verbose=True):

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    files = os.listdir(data_dir)
    files.sort()

    filepaths_input = [os.path.join(data_dir, x) for x in files]
    filepaths_output = [os.path.join(output_dir, x) for x in files]
    
    if verbose:
        print(f'Number of files:         {len(filepaths_input)}')
        print(f'Sample filepath input:   {filepaths_input[0]}')
        print(f'Sample filepath output:  {filepaths_output[0]}')
              
    return filepaths_input, filepaths_output


def read_in_routine(filepath_input, fs_manual=10):

    datatype = filepath_input.split('.')[-1]

    assert datatype in ['csv', 'h5'], 'Currently, .csv or .h5 files are expected to be read in.'

    if datatype == 'csv':

        data = pd.read_csv(filepath_input)
        data.columns = [x.lower() for x in data.columns]
        data.datetime = pd.to_datetime(data.datetime, infer_datetime_format=1)
        data.set_index('datetime', inplace=True)
        data.rename({'band_smooth': 'movavg_0_5s'}, axis=1, inplace=True)
        hdr = {}
        
    if datatype == 'h5':
        
        data, hdr = load_sleep_data(filepath_input, idx_to_datetime=1)
#         data[np.isinf(data)] = np.nan
#         data = data.astype('float64')         
        fs = hdr['fs']
        data.columns = [x.lower() for x in data.columns]
        data.rename({'band_smooth': 'movavg_0_5s'}, axis=1, inplace=True)
        
        hdr['study_id'] = np.int32(hdr['study_id'])
        hdr['MRN'] = np.int32(hdr['MRN'])
        hdr['fs'] = np.int32(hdr['fs'])
        hdr['start_datetime'] = np.array([hdr['start_datetime'].year,hdr['start_datetime'].month,hdr['start_datetime'].day, hdr['start_datetime'].hour, hdr['start_datetime'].minute, hdr['start_datetime'].second, hdr['start_datetime'].microsecond])

    if not 'band_unscaled' in data:
        print('Airgo signal not yet normalized. Do normalization on band and movavg_0_5s, keep original: band_unscaled')
        data = renormalize_airgo(data)
    
    if 'fs' in hdr:
        fs = hdr['fs']
    else:
        fs = fs_manual
            
    return data, hdr, fs


def save_data_routine(data, filepath_output, hdr=None, overwrite=False):
    
    datatype = filepath_input.split('.')[-1]
    assert datatype in ['csv', 'h5'], 'Currently, .csv or .h5 expected.'
    
    base_path = os.path.basename(filepath_output)
    if not os.path.exists(base_path):
        os.makedirs(base_path)
    
    if datatype == 'csv':
        data.to_csv(filepath_output)

    if datatype == 'h5':
        write_to_hdf5_file(data, filepath_output, hdr=hdr, overwrite=overwrite)

        
def airgo_resample_preprocess(data, seconds='0.1', interpolation_limit = 15):
    
    data = data.resample(f'{seconds}S').mean()
    data = data.interpolate(method='pchip', order=3, limit_area='inside', limit=interpolation_limit)
    return data

def clip_z_normalize(signal):

    signal_clipped = np.clip(signal.dropna(), np.percentile(signal.dropna(),1), np.percentile(signal.dropna(),99))
    mean = np.mean(signal_clipped.astype(np.float32))
    std = np.std(signal_clipped.astype(np.float32))
    signal = (signal - mean)/std

    return signal

def renormalize_airgo(data, fs=10):
    
    data['band_unscaled'] = data['band'].copy()
    data['band'] = clip_z_normalize(data.band)
    if 'movavg_0_5s' in data.columns:
        data.drop(['movavg_0_5s'], axis=1, inplace=True)
    data['movavg_0_5s'] = data['band'].reset_index()['band'].rolling(int(0.5*fs), center=True, min_periods=1).mean().values
    return data

### Compute all airgo features (RR, MV, instability, complexity, actigraphy), apply apnea detection and sleep staging model. <br>  If other signals are available, like SpO2 (bedmaster, PSG), then models that use these signals will also be utilized.
Currently models:
Apnea Detection: Respiration Only (needs features computed), Respiration+SpO2.  
Sleep Staging: Based on Respiration Only, Actigraphy Only.

### input: 
<b> list of paths to files. </b>  
files: either original airgo files, or merged with bedmaster or PSG data.

In [3]:
data_dir = '/media/ssd2/sleeplab_final_files'
output_dir = '/media/ssd2/sleeplab_final_files'

overwrite = True

filepaths_input, filepaths_output = filepaths_routine(data_dir, output_dir)

Number of files:         412
Sample filepath input:   /media/ssd2/sleeplab_final_files/psg_airgo_10hz_001.h5
Sample filepath output:  /media/ssd2/sleeplab_final_files/psg_airgo_10hz_001.h5


### Params / Settings

In [4]:
fs_manual = 10
do_resample_and_interpolation = False     # recommended for raw airgo, resampling to 'perfect 10Hz'
do_compute_airgo_features = False            # all features. by default, complexity features are computed in this code which are the slowest but needed for apnea prediction.
do_apply_apnea_prediction_models = False     # respiration-only and respiration+spo2 models.
do_apply_sleep_staging_models = True        # respiration-only, actigraphy-only.

In [5]:
# GET FOLD INFOS:
fold_dir = '/home/wolfgang/repos/AirGoSleepPyT0/fold_ids/'

if 0:
    
    folds = pd.DataFrame()

    for fold_id in range(5):
        fold_tmp = pd.read_csv(os.path.join(fold_dir, f'test_files_fold{fold_id}.csv'))
        fold_tmp['fold_id'] = fold_id
        folds = pd.concat([folds, fold_tmp])

    folds.to_csv(os.path.join(fold_dir, f'all_test_folds.csv'), index=False)
    
else:
    folds = pd.read_csv(os.path.join(fold_dir, f'all_test_folds.csv'))
    
folds.fileID = [x.replace('Feature_Air','psg_airgo_10hz_').replace('.mat','.h5') for x in folds.fileID]
print(folds.head(3))

                  fileID  fold_id
0  psg_airgo_10hz_002.h5        0
1  psg_airgo_10hz_013.h5        0
2  psg_airgo_10hz_025.h5        0


In [6]:
for fold_id in range(5):
#     fold_id = 0
    print(f'Fold {fold_id}')
    # files for fold
    files = folds.loc[folds.fold_id == fold_id, 'fileID']
    file_endings_fold = files.apply(lambda x: x[-6:]).values

Fold 0
Fold 1
Fold 2
Fold 3
Fold 4


In [7]:
# filepath_input = filepaths_input[0]
# filepath_output = filepaths_output[0]

for fold_id in range(5):

    models_dir = '/media/mad3/Projects/AirGoSleepStaging/final_models/ActivityIndex10sec'

    for filepath_input, filepath_output in list(zip(filepaths_input, filepaths_output)):

#         try:
        if 1:
#             import pdb; pdb.set_trace()
            files = folds.loc[folds.fold_id == fold_id, 'fileID']
            file_endings_fold = files.apply(lambda x: x[-6:]).values
            if not filepath_input[-6:] in file_endings_fold:
                continue

            if (os.path.exists(filepath_output)) & (not overwrite):
                print(f'{filepath_output} already exists. Continue.')
                continue

            print(filepath_input)

            data, hdr, fs = read_in_routine(filepath_input, fs_manual=fs_manual)

            # skip missing airgo:
            if not 'band' in data.columns:
                print('No AirGo data at all.')
                continue

            if data.acc_ai_10sec.dropna().mean() > 1:
                print(file_tmp)
                print('data.acc_ai_10sec.dropna().mean() > 1 --> V4.')
                continue

            if any(data.band_unscaled.dropna()>5000):
                print(file_tmp)
                print('unscaled airgo amplitude>5000 --> V4.')
                continue


            if do_resample_and_interpolation:
                data = airgo_resample_preprocess(data)

            if do_compute_airgo_features:
                data = compute_airgo_features(data, fs=fs, complexity_features=1)

            if do_apply_apnea_prediction_models:
                data = apply_apnea_prediction_models(data, fs=fs)

            if do_apply_sleep_staging_models:
                
                data.drop(['stage_pred_activity1sec'], axis=1, inplace=True)
                data.drop(['stage_pred_vcomb1'], axis=1, inplace=True)


                data = apply_airgo_sleep_staging_models_fold(data, fs=fs, model_ids=['activity10sec'], 
                                                             input_signals=['acc_ai_10sec'], models_dir=models_dir, 
                                                             fold_id=fold_id)


            save_data_routine(data, filepath_output, hdr=hdr, overwrite=overwrite)

#         except Exception as e:
#             print('Exception for ' + filepath_input)
#             print(e)
#             continue

/media/ssd2/sleeplab_final_files/psg_airgo_10hz_002.h5
Airgo signal not yet normalized. Do normalization on band and movavg_0_5s, keep original: band_unscaled
activity10sec


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/torch/serialization.py:657: SourceChangeWarning: source code of class 'mymodel.CHESTSleepNet' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)
/home/wolfgang/repos/sleep_research_io/sleep_research_functions.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[signal_tmp][pd.isna(data[signal_tmp])] = -1
/home/wolfgang/repos/sleep_research_io/sleep_research_functions.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guid

/media/ssd2/sleeplab_final_files/psg_airgo_10hz_013.h5
Airgo signal not yet normalized. Do normalization on band and movavg_0_5s, keep original: band_unscaled
activity10sec
/media/ssd2/sleeplab_final_files/psg_airgo_10hz_025.h5
Airgo signal not yet normalized. Do normalization on band and movavg_0_5s, keep original: band_unscaled
activity10sec
/media/ssd2/sleeplab_final_files/psg_airgo_10hz_033.h5
Airgo signal not yet normalized. Do normalization on band and movavg_0_5s, keep original: band_unscaled
activity10sec
/media/ssd2/sleeplab_final_files/psg_airgo_10hz_034.h5
Airgo signal not yet normalized. Do normalization on band and movavg_0_5s, keep original: band_unscaled
activity10sec
/media/ssd2/sleeplab_final_files/psg_airgo_10hz_053.h5
Airgo signal not yet normalized. Do normalization on band and movavg_0_5s, keep original: band_unscaled
activity10sec
/media/ssd2/sleeplab_final_files/psg_airgo_10hz_055.h5
Airgo signal not yet normalized. Do normalization on band and movavg_0_5s, keep 

In [38]:
plt.subplots(sharex=True, figsize=(9,9))
ax1 = plt.subplot(611)
ax1.plot(data.band)
ax2 = plt.subplot(612, sharex=ax1)
ax2.plot(data.stage_pred_amendsumeffort_0)
ax3 = plt.subplot(613, sharex=ax1)
ax3.plot(data.stage_pred_amendsumeffort)
ax4 = plt.subplot(614, sharex=ax1)
ax4.plot(data.stage_pred_activity1sec)
ax5 = plt.subplot(615, sharex=ax1)
ax5.plot(data.stage_pred_activity10sec)
ax5 = plt.subplot(616, sharex=ax1)
ax5.plot(data.acc_ai_10sec)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [11]:
# plt.subplots(3, sharex=True)
# ax1 = plt.subplot(511)
# ax1.plot(data.band)
# ax2 = plt.subplot(512, sharex=ax1)
# ax2.plot(data.apnea_prob_RO_a3.dropna())
# ax3 = plt.subplot(513, sharex=ax1)
# ax3.plot(data.stage_pred_amendsumeffort)
# ax4 = plt.subplot(514, sharex=ax1)
# ax4.plot(data.stage_pred_activity1sec)
# ax5 = plt.subplot(515, sharex=ax1)
# ax5.plot(data.acc_ai_1sec)

In [7]:
data.columns

Index(['accx', 'accy', 'accz', 'band', 'band_unscaled', 'movavg_0_5s',
       'movavg_1_2s', 'movavg_1min', 'deriv1', 'ventilation0',
       'ventilation_3s', 'ventilation_8s', 'ventilation_10s',
       'ventilation_12s', 'ventilation_15s', 'ventilation_30s',
       'ventilation_60s', 'ventilation_5min', 'ventilation_10min',
       'ventilation_10s_smooth', 'hypo_10_60', 'movmedian_5min',
       'movmedian_10min', 'movstd_8s', 'movstd_12s', 'movstd_15s',
       'movstd_10s', 'movstd_30s', 'movstd_60s', 'movstd_5min', 'movstd_10min',
       'katz_fd_10s_smoothed', 'katz_fd_30s_smoothed', 'katz_fd_45s_smoothed',
       'katz_fd_60s_smoothed', 'sample_entropy_10s_smoothed',
       'sample_entropy_30s_smoothed', 'sample_entropy_60s_smoothed',
       'movavg_0_5s_unscaled', 'peaks', 'movstd_1min_unscaled',
       'movstd_30sec_unscaled', 'rr_10s', 'rr_60s', 'rr_10s_smooth',
       'movmedian_1min', 'movmedian_30sec', 'IBI', 'IBI_mean_5min',
       'IBI_std_5min', 'ventilation_5min_deriv', '